# Ingest constructors.json file
1. Read the file using spark dataframe reader API
1. Add Metadata Columns 
    - Source File
    - Ingestion Timestamp
1. Write to bronze delta table

In [0]:
%run  ../common/01_Environmnet_config

In [0]:
%run  ../common/02_bronze_helpers

In [0]:
source_File = f"{landing_folder_path}/constructors.json"
table_name = f"{catalog_name}.{bronze_schema}.constructors"
print(source_File)
print(table_name)

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

constructors_schema = StructType([
    StructField('constructorId', StringType(), True),
    StructField('name', StringType(), True),
    StructField('nationality', StringType(), True),
    StructField('url', StringType(), True)
])

In [0]:
constructor_df = (
    spark.read.format("json")
    .schema(constructors_schema)
    .option("mode", "FAILFAST")
    .load(source_File)
)

In [0]:
display(constructor_df)

In [0]:
constructor_final_df = add_ingestion_medatat(constructor_df)

In [0]:
(
    constructor_final_df
                .write
                .format("delta")
                .mode("overwrite")
                .saveAsTable(table_name)
    
)

In [0]:
spark.sql(F"select * from {table_name}").display()